# SDDP with SDDP.jl: hydro-thermal examples

The slides *07. SDDP* describe the algorithm, and the notebook *SDDP_from_scratch* writes it step by
step. Here we use a production implementation, [SDDP.jl](https://github.com/odow/SDDP.jl) (Dowson and
Kapelevich, 2021), on two small hydro-thermal problems: we only describe the subproblem of a stage
and its random data, and the package runs the forward and backward passes.

- A model is a **policy graph**: one node per stage, each with its JuMP subproblem.
- A variable declared with `SDDP.State` has an incoming value `x.in`, the trial decision
  $\hat{x}_{t-1}$ of the slides, and an outgoing value `x.out`, the decision $x_t$ passed on to the
  next stage.
- `SDDP.parameterize` lists the realizations of $\xi_t$, stagewise independent.
- `lower_bound` bounds $\theta_t$ from below, which keeps the subproblems bounded until their
  first cut.
- In the log of `SDDP.train`, *simulation* is the cost $z_i$ of the scenario drawn by the forward
  pass (one per iteration, $K = 1$), and *bound* is the lower bound $z_{LB}$, the value of the
  first-stage problem.

## 1. FAST: a two-stage hydro-thermal problem

Adapted from the [FAST example of SDDP.jl](https://odow.github.io/SDDP.jl/latest/examples/FAST_hydro_thermal/),
itself taken from [FAST](https://github.com/leopoldcambier/FAST/blob/master/examples/hydro%20thermal/hydro_thermal.m).

Electricity is produced either by a thermal plant or by a hydroelectric plant. The demand is 6 at
each of the two stages; fuel costs 5 per unit of demand, and water is free. The reservoir is empty
at the start and holds at most 8 units; the rainfall is 6 at stage 1, and at stage 2 it is low (2)
or high (10), with probability $1/2$. Let $x_1$ be the water left in the reservoir at the end of
stage 1, $y_1$ the water used, and $p_1$ the fuel burnt. The problem of stage 1 is
\begin{align*}
  \min\ & 5 p_1 + \mathcal{Q}_2(x_1) \\
  \text{s.t. } & x_1 \leq 8, \quad x_1 \leq 6 - y_1, \quad p_1 + y_1 \geq 6, \quad x_1, y_1, p_1 \geq 0,
\end{align*}
where $6 - y_1$ is the initial level $0$, plus the rainfall $6$, minus the water used. The expected
cost-to-go can be found by hand: at stage 2, the plant uses the available water up to the demand,
$y_2 = \min \{ x_1 + \xi_2, 6 \}$, and burns fuel for the rest, so that
\begin{align*}
\mathcal{Q}_2(x_1) &= \mathbb{E}\left[ 30 - 5 \min \{ x_1 + \xi_2, 6 \} \right]
  = 30 - \tfrac{1}{2} \left( 5 \min \{ x_1 + 2, 6 \} + 5 \min \{ x_1 + 10, 6 \} \right) \\
  &= 15 - \tfrac{5}{2} \min \{ x_1 + 2, 6 \}.
\end{align*}
Storing a unit of water costs 5 in fuel at stage 1, and saves at most $5/2$ at stage 2. The solution
is therefore
$$
   x_1 = 0,\ y_1 = 6,\ p_1 = 0,
$$
for an expected cost of $\mathcal{Q}_2(0) = 10$.

In [1]:
using SDDP, HiGHS
using Random, Statistics, Test

In [2]:
model = SDDP.PolicyGraph(
    SDDP.LinearGraph(2),      # two stages
    lower_bound = 0.0,        # bound on θ: the costs are nonnegative
    optimizer = HiGHS.Optimizer,
) do sp, t
    # the state is the reservoir level: x.in at the start of the stage, x.out at its end
    @variable(sp, 0 <= x <= 8, SDDP.State, initial_value = 0.0)
    @variables(sp, begin
        y >= 0                # water used
        p >= 0                # fuel burnt
        ξ                     # rainfall, fixed by parameterize
    end)
    @constraints(sp, begin
        p + y >= 6            # demand
        x.out <= x.in - y + ξ # water balance; the excess is spilled
    end)
    # the rainfall is deterministic at stage 1, and takes one of two values at stage 2
    RAINFALL = (t == 1 ? [6] : [2, 10])
    SDDP.parameterize(sp, RAINFALL) do ω
        JuMP.fix(ξ, ω)
    end
    @stageobjective(sp, 5 * p)
end

A policy graph with 2 nodes.
 Node indices: 1, 2


The deterministic equivalent is the extensive form of the slides. With two scenarios it is tiny
(the names are repeated across the nodes), and it confirms the value 10.

In [3]:
det = SDDP.deterministic_equivalent(model, HiGHS.Optimizer)
println(det)
set_silent(det)
JuMP.optimize!(det)
@test JuMP.objective_value(det) ≈ 10

Min 5 p#1 + 2.5 p#2 + 2.5 p#2


Subject to
 

x_out#1 - x_in#2 = 0
 x_out#1 - x_in#2 = 0
 y#1 + p#1 ≥ 6
 y#2 + p#2 ≥ 6
 y#2 + p#2 ≥ 6
 -x_in#1 + x_out#1 + y#1 - ξ#1 ≤ 0
 -x_in#2 + x_out#2 + y#2 - ξ#2 ≤ 0
 -x_in#2 + x_out#2 + y#2 - ξ#2 ≤ 0
 x_in#1 = 0
 ξ#1 = 6
 ξ#2 = 2
 ξ#2 = 10
 x_out#1 ≥ 0
 y#1 ≥ 0
 p#1 ≥ 0
 _[6]#1 ≥ 0
 x_out#2 ≥ 0
 y#2 ≥ 0
 p#2 ≥ 0
 _[6]#2 ≥ 0
 x_out#2 ≥ 0
 y#2 ≥ 0
 p#2 ≥ 0
 _[6]#2 ≥ 0
 x_out#1 ≤ 8
 x_out#2 ≤ 8
 _[6]#2 ≤ 0
 x_out#2 ≤ 8
 _[6]#2 ≤ 0



Test Passed

SDDP finds the same value. The simulated cost is 20 when the rainfall is low and 0 when it is high:
each scenario is far from the expected cost of the policy, 10, which is why the statistical upper bound
of the slides needs many scenarios.

In [4]:
Random.seed!(1234)
SDDP.train(model, iteration_limit = 10, log_frequency = 5)
@test SDDP.calculate_bound(model) ≈ 10

-------------------------------------------------------------------


         SDDP.jl (c) Oscar Dowson and contributors, 2017-26
-------------------------------------------------------------------
problem


  nodes           : 2
  state variables : 1
  scenarios       : 2.00000e+00
  existing cuts   : false
options
  solver          : serial mode
  risk measure    : SDDP.

Expectation()
  sampling scheme : SDDP.InSampleMonteCarlo
subproblem structure
  VariableRef                             : [6, 6]
  AffExpr in MOI.GreaterThan{Float64}     : [1, 1]
  AffExpr in MOI.LessThan{Float64}        : [1, 1]
  VariableRef in MOI.EqualTo{Float64}     : [1, 1]
  VariableRef in MOI.GreaterThan{Float64} : [4, 4]
  VariableRef in MOI.LessThan{Float64}    : [1, 2]
numerical stability report
  matrix range     [1e+00, 1e+00]
  objective range  [1e+00, 5e+00]
  bounds range     [8e+00, 8e+00]
  rhs range        [6e+00, 6e+00]


-------------------------------------------------------------------
 iteration    simulation      bound        time (s)     solves  pid
-------------------------------------------------------------------
         5

   0.000000e+00  1.000000e+01  4.659515e+00        25   1
        10   0.000000e+00  1.000000e+01  4.795463e+00        50   1
-------------------------------------------------------------------
status         : iteration_limit
total time (s) : 4.795463e+00
total solves   : 50
best bound     :  1.000000e+01
numeric issues : 0
-------------------------------------------------------------------



Test Passed

## 2. Hydro-thermal scheduling over an infinite horizon

Adapted from the [SDDP.jl documentation](https://odow.github.io/SDDP.jl/latest/examples/Hydro_thermal/#Hydro-thermal-scheduling).
This example goes beyond the slides: its policy graph has a cycle, which models an infinite horizon.

### Problem description

In a hydro-thermal problem, the agent controls a hydro-electric generator and reservoir. Each time
period, they choose a generation quantity from thermal, $g_t$, and from hydro, $g_h$, in order to meet
the demand $w_d$, a stagewise-independent random variable. The state variable, $x$, is the quantity of
water in the reservoir, with a minimum level of 5 units and a maximum level of 15 units. There are 10
units of water in the reservoir at the start of time, so that $x_0 = 10$. The state variable is
connected through time by the water balance constraint
$$x.out = x.in - g_h - s + w_i,$$
where $x.out$ is the quantity of water at the end of the time period, $x.in$ the quantity of water at
its start, $s$ the quantity of water spilled from the reservoir, and $w_i$ a stagewise-independent
random variable, the inflow into the reservoir during the time period.

There are three stages, $t = 1, 2, 3$, representing summer-fall, winter and spring, and the problem
is solved over an infinite horizon, with a discount factor of 0.95. The discount is modelled as the
probability 0.95 of going on from spring to a new summer-fall: the expected discounted cost is the
expected cost over a random number of years.

In each stage, the agent incurs the cost of spillage, plus the cost of thermal generation, which
depends on the stage $t = 1, 2, 3$. In each stage, $w$ is drawn from the set
$(w_i, w_d) \in \{(0, 7.5), (3, 5), (10, 2.5)\}$ with equal probability.

### Constructing the policy graph

There are three stages in our problem, so we construct a linear policy graph with three stages using
`SDDP.LinearGraph`:

In [5]:
graph = SDDP.LinearGraph(3)

Root
 0
Nodes
 1
 2
 3
Arcs
 0 => 1 w.p. 1.0
 1 => 2 w.p. 1.0
 2 => 3 w.p. 1.0

Then, because we want to solve an infinite-horizon problem, we add an edge from node 3 back to node 1, with probability 0.95:

In [6]:
SDDP.add_edge(graph, 3 => 1, 0.95)

### Constructing the model

Compared to a JuMP model, one key difference is that we use `@stageobjective` instead of `@objective`.
The function `SDDP.parameterize` takes a list of supports for $w$, and parameterizes the JuMP model `sp`
by setting the right-hand sides of the appropriate constraints (note how the constraints initially have
a right-hand side of 0). By default, the realizations are equally likely, but a probability vector can
also be provided.

In [7]:
model = SDDP.PolicyGraph(
    graph,
    sense = :Min,
    lower_bound = 0.0,
    optimizer = HiGHS.Optimizer,
) do sp, t
    @variable(sp, 5 <= x <= 15, SDDP.State, initial_value = 10)
    @variable(sp, g_t >= 0)
    @variable(sp, g_h >= 0)
    @variable(sp, s >= 0)
    @constraint(sp, balance, x.out - x.in + g_h + s == 0)
    @constraint(sp, demand, g_h + g_t == 0)
    @stageobjective(sp, s + t * g_t)
    SDDP.parameterize(sp, [[0, 7.5], [3, 5], [10, 2.5]]) do w
        set_normalized_rhs(balance, w[1])
        return set_normalized_rhs(demand, w[2])
    end
end

A policy graph with 3 nodes.
 Node indices: 1, 2, 3


### Training the policy

`SDDP.train` runs the forward and backward passes. There is no statistical stopping test here:
training stops at the iteration limit, as most applications do (see the end of *SDDP_from_scratch*).

In [8]:
Random.seed!(1234)
SDDP.train(model, iteration_limit = 200, log_frequency = 25)

-------------------------------------------------------------------
         SDDP.jl (c) Oscar Dowson and contributors, 2017-26
-------------------------------------------------------------------
problem
  nodes           : 3
  state variables : 1
  scenarios       : Inf
  existing cuts   : false
options
  solver          : serial mode
  risk measure    : SDDP.Expectation()
  sampling scheme : SDDP.InSampleMonteCarlo
subproblem structure
  VariableRef                             : [6, 6]
  AffExpr in MOI.EqualTo{Float64}         : [2, 2]
  VariableRef in MOI.GreaterThan{Float64} : [5, 5]
  VariableRef in MOI.LessThan{Float64}    : [1, 1]
numerical stability report
  matrix range     [1e+00, 1e+00]
  objective range  [1e+00, 3e+00]
  bounds range     [5e+00, 2e+01]
  rhs range        [2e+00, 1e+01]


-------------------------------------------------------------------
 iteration    simulation      bound        time (s)     solves  pid
-------------------------------------------------------------------
        25

   8.587120e+01  2.335361e+02  8.085811e-01      5907   1
        50

   7.307086e+02  2.362065e+02  1.733345e+00     12954   1
        75

   8.850357e+01  2.363826e+02  2.610871e+00     18009   1
       100

   2.500000e+00  2.364299e+02  4.288699e+00     25548   1
       125

   2.065012e+02  2.364346e+02  6.224130e+00     30891   1
       150

   6.165010e+02  2.364363e+02  1.063337e+01     38406   1
       175

   1.415000e+02  2.364365e+02  1.464559e+01     43725   1
       200

   7.000000e+01  2.364366e+02  1.773112e+01     48168   1
-------------------------------------------------------------------
status         : iteration_limit
total time (s) : 1.773112e+01
total solves   : 48168
best bound     :  2.364366e+02
numeric issues : 0
-------------------------------------------------------------------



### Simulating the policy

After training, `SDDP.simulate` simulates the policy. The mean cost of the simulations estimates the
expected cost of the policy, the upper bound of the slides; with a horizon of random length, the costs
vary a lot.

In [9]:
sims = SDDP.simulate(model, 1000, [:g_t])
mu = round(mean([s[1][:g_t] for s in sims]), digits = 2)
println("On average, $(mu) units of thermal are used in the first stage.")

costs = [sum(stage[:stage_objective] for stage in s) for s in sims]
half = 1.96 * std(costs) / sqrt(length(costs))
println("Simulated cost: ", round(mean(costs), digits = 1), " ± ", round(half, digits = 1),
        " (95% CI); lower bound: ", round(SDDP.calculate_bound(model), digits = 1))

On average, 2.17 units of thermal are used in the first stage.


Simulated cost: 242.7

 ± 16.0 (95% CI); lower bound: 236.4


### Extracting the water values

Finally, `SDDP.ValueFunction(model[1])` extracts the cuts of node 1, that is, the approximation
$\theta_1$ of the expected cost-to-go as a function of the water left at the end of stage 1, and
`SDDP.evaluate` returns its value and its slope at a given level. Since we are minimizing, the
price of water is negative: each additional unit of water decreases the expected long-run cost.

In [10]:
V = SDDP.ValueFunction(model[1])
cost, price = SDDP.evaluate(V, x = 10)

(233.56183571468856, Dict(:x => -0.6591550826539746))